### Firewall API - Network Zones Management

[![Python 3.11+](https://img.shields.io/badge/Python-3.11%2B-blue.svg)](https://www.python.org/downloads/)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](../LICENSE)
[![Development Status: Beta 0.8.0](https://img.shields.io/badge/Status-Beta%200.8.0-orange.svg)](https://pypi.org/project/sfos-api/)
[![Code Style: Ruff](https://img.shields.io/endpoint?url=https://raw.githubusercontent.com/astral-sh/ruff/main/assets/badge/v2.json)](https://github.com/astral-sh/ruff)

**Purpose:** Create, read, update, and delete network security zones and configure zone device access services.

**Topics:**
- Basic CRUD operations using domain StrEnum objects (`Network`)
- Managing LAN and DMZ zones and appliance access permissions (`Network.ZONES`)
- Filtering and auto-pagination
- Error handling
- Bulk operations

**Requirements:**
- Credentials configured (`.credentials/config.ini`)
- SFOS firewall accessible
- API key with appropriate permissions

---

#### Setup: Import Libraries and Configure Client

In [ ]:
from utils_api import find_credentials_file, print_data

from sfos_api import Network, SFOSClient

# Initialize the SFOS API Client
client = SFOSClient(config_path=find_credentials_file())
print("✅ SFOS API client initialized")

#### Create Network Zones

Create network zone objects using the `Network.ZONES` endpoint object.

Zones are used to group network interfaces and define traffic policies.

API Link: [Create a zone | Sophos Firewall API](https://docs.sophos.com/nsg/sfos-api/rest-api/api/firewall-config-openapi/create-zone/)

---

In [ ]:
# Create a network zone
endpoint = Network.ZONES
zone_name = "test_zone_dmz"

payload = {
    "name": zone_name,
    "description": "DMZ zone for hosting services",
    "type": "dmz",
}

try:
    response = client.post(endpoint, payload)
    print(f"✅ Created zone '{zone_name}'")
    print_data(response)
except Exception as e:
    print(f"Error creating zone: {e}")
    print_data(str(e))

In [ ]:
# Create another network zone
endpoint = Network.ZONES
zone_name = "test_zone_guest"

payload = {
    "name": zone_name,
    "description": "Guest network zone",
    "type": "lan",
}

try:
    response = client.post(endpoint, payload)
    print(f"✅ Created zone '{zone_name}'")
    print_data(response)
except Exception as e:
    print(f"Error creating zone: {e}")
    print_data(str(e))

#### Read Network Zones

Retrieve zone information by ID/name or list all zones.

API Links: [Get all zones | Sophos Firewall API](https://docs.sophos.com/nsg/sfos-api/rest-api/api/firewall-config-openapi/get-all-zones/) | [Get a zone | Sophos Firewall API](https://docs.sophos.com/nsg/sfos-api/rest-api/api/firewall-config-openapi/get-zone/)

---

In [ ]:
# Get a specific zone by name
endpoint = Network.ZONES
zone_name = "test_zone_guest"

try:
    response = client.get(endpoint, id_or_name=zone_name)
    print(f"✅ Retrieved zone '{zone_name}'")
    print_data(response)
except Exception as e:
    print(f"Error reading zone: {e}")
    print_data(str(e))

In [ ]:
# Get all zones
endpoint = Network.ZONES

try:
    response = client.get(endpoint)

    if "items" in response and response["items"]:
        print(f"✅ Retrieved {len(response['items'])} zones")
        print_data(response["items"], include=["name", "description", "type"])
    else:
        print("No zones found")
except Exception as e:
    print(f"Error reading zones: {e}")
    print_data(str(e))

In [ ]:
# Get zones with filtering
endpoint = Network.ZONES
params = {
    "nameContains": "test",
}

try:
    response = client.get(endpoint, params=params)

    if "items" in response:
        print(f"✅ Retrieved {len(response['items'])} zones containing 'test-'")
        print_data(response["items"], include=["name", "description", "type", "services"])
    else:
        print("No zones found matching filter")
except Exception as e:
    print(f"Error reading zones: {e}")
    print_data(str(e))

#### Update Network Zone

Modify an existing zone's properties.

API Link: [Update a zone | Sophos Firewall API](https://docs.sophos.com/nsg/sfos-api/rest-api/api/firewall-config-openapi/update-zone/)

---

In [ ]:
# Update a zone
endpoint = Network.ZONES
zone_name = "test_zone_dmz"

payload = {
    "description": "Updated DMZ zone - for web and app servers",
    "services": [
        {"name": "ssh"},
        {"name": "https"},
    ],
}

try:
    response = client.patch(endpoint, id_or_name=zone_name, data=payload)
    print(f"✅ Updated zone '{zone_name}'")
    print_data(response)
except Exception as e:
    print(f"Error updating zone: {e}")
    print_data(str(e))

#### Delete Network Zones

Remove zone objects (single or bulk delete).

API Links: [Delete a zone | Sophos Firewall API](https://docs.sophos.com/nsg/sfos-api/rest-api/api/firewall-config-openapi/delete-zone/) | [Bulk delete zones | Sophos Firewall API](https://docs.sophos.com/nsg/sfos-api/rest-api/api/firewall-config-openapi/bulk-delete-zones/)

---

In [ ]:
# Delete a single zone
endpoint = Network.ZONES
zone_name = "test_zone_dmz"

try:
    response = client.delete(endpoint, id_or_name=zone_name)
    print(f"✅ Deleted zone '{zone_name}'")
    if response:
        print_data(response)
except Exception as e:
    print(f"Error deleting zone: {e}")
    print_data(str(e))

In [ ]:
# Bulk delete zones
endpoint = Network.ZONES

items_to_delete = [
    {"name": "test_zone_dmz"},
    {"name": "test_zone_guest"},
]

try:
    response = client.delete(endpoint, items=items_to_delete)
    print(f"✅ Bulk deleted {len(items_to_delete)} zones")
    if response:
        print_data(response)
except Exception as e:
    print(f"Error bulk deleting zones: {e}")
    print_data(str(e))

In [ ]:
# Bulk delete zones with filter
endpoint = Network.ZONES

try:
    response = client.delete(endpoint, nameContains="test_")
    print("✅ Bulk deleted zones matching nameContains='test_'")
    if response:
        print_data(response)
except Exception as e:
    print(f"Error bulk deleting zones: {e}")
    print_data(str(e))

In [ ]:
# Close the client session when done
client.close()
print("✅ Client session closed")